In [1]:
%run 01_MNESIS_boilerplate.ipynb

datetag = '2026-04-21'
SNNtorch version 0.9.4


In [2]:
# %whos

# MNESIS: defining parameters



In [ ]:
# https://docs.python.org/3/library/dataclasses.html?highlight=dataclass#module-dataclasses
from dataclasses import dataclass, asdict, field

DEBUG = 4 # to speed up simulations
DEBUG = 1 # production

@dataclass
class Params:
    datetag: str = datetag

    N_neuron: int = 1024//DEBUG # number of presynaptic inputs
    # N_neuron: int = 512 # number of presynaptic inputs
    num_delay: int = 41 # number of timesteps in SM, must be a odd number for convolutions
    # N_pattern: int = 8 # number of spiking motifs
    N_pattern: int = 16 # number of spiking motifs
    ## Raster plots
    N_time: int =  1000//DEBUG # number of timebins for the WM patterns
    N_pretime: int =  50 # number of timebins for spontaneous activity before and after the stimulus
    # p_A: float = .00016 # prior probability of firing for postsynaptic raster plot (spike per timebin)
    p_A: float = .001 # prior probability of firing for postsynaptic raster plot (spike per timebin)
    seed: int = 2018 # seed
    device = device

    # network
    lif_beta: float = .7
    lif_threshold: float = .8
    # learn_beta: bool = True
    learn_beta: bool = False
    learn_threshold: bool = False
    # learn_threshold: bool = True
    do_pinv: bool = False
    do_deconv: bool = True
    
    # learning
    num_epochs: int = 16 #2**6
    num_warmup_epochs: int = 16 # 2**4
    base_lr: float = 1.0e-3
    final_lr: float = 100.e-6
    delta1: float = 10.e-3
    delta2: float =  50.e-6
    weight_decay: float = 1.e-9
    dropout: float = 0.37
    alpha_surrogate: float = 15.
    surrogate_name: str = "FastSigmoid"
    loss_name: str = "SpikeF1scoreLoss" # 'MSELoss' # 'L1Loss' #
    reset_mechanism: str = "subtract" # "zero" # 
    optimizer: str = 'adamw' # 'sgd' # 'adam' #


    ## figures
    verbose: bool = False # Displays more verbose output.
    fig_width: float = 15 # width of figure
    fig_height: float = 9 # width of figure
    phi: float = 1.61803 # beauty is gold
    N_time_show: int = 400 # number of time points to show in plots
    N_neuron_show: int = 128 # number of SM to show in plots
    N_scan: int = 17//DEBUG +1 # number of values to scan


    def __post_init__(self):
        torch.manual_seed(self.seed)
        np.random.seed(self.seed)


In [4]:
opt = Params()
opt

Params(datetag='2026-04-21', N_neuron=1024, num_delay=41, N_pattern=16, N_time=1000, N_pretime=50, p_A=0.001, seed=2018, lif_beta=0.7, learn_beta=False, learn_threshold=False, do_pinv=False, do_deconv=True, num_epochs=16, num_warmup_epochs=16, base_lr=0.001, final_lr=0.0001, delta1=0.01, delta2=5e-05, weight_decay=1e-09, init_gain=1.5, dropout=0.37, alpha_surrogate=15.0, surrogate_name='FastSigmoid', loss_name='SpikeF1scoreLoss', reset_mechanism='subtract', optimizer='sgd', verbose=False, fig_width=15, fig_height=9, phi=1.61803, N_time_show=400, N_neuron_show=128, N_scan=5)

In [5]:
print(f'Spikes in one target {opt.p_A * opt.N_neuron * opt.N_time:.1f},  in a SM window {opt.p_A * opt.N_neuron * opt.num_delay:.1f}')

Spikes in one target 1024.0,  in a SM window 42.0


In [6]:
print(f'for a value {opt.lif_beta=:.1f}, the time constant is {- 1 / np.log(opt.lif_beta):.1f} steps')

for a value opt.lif_beta=0.7, the time constant is 2.8 steps
